In [6]:
import requests
import csv
import os
from dotenv import load_dotenv
import json

url = "https://app.omie.com.br/api/v1/produtos/nfconsultar/"

# Carrega o .env
load_dotenv()
app_key = os.getenv("app_key")
app_secret = os.getenv("app_secret")

# Abrir um arquivo CSV para escrita
with open('produtos.csv', mode='w', newline='', encoding='utf-8') as csv_file:
    fieldnames = ['NCM', 'uCOM', 'qCom', 'xProd']
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    writer.writeheader()
    
    # Variável para controlar a paginação
    pagina = 1
    total_de_paginas = None
    
    while True:
        # Criar payload para a página atual
        payload = {
            "call": "ListarNF",
            "app_key": app_key,
            "app_secret": app_secret,
            "param": [{
                "pagina": pagina,
                "registros_por_pagina": 100,
                "ordenar_por": "CODIGO"
            }]
        }
        
        # Fazer a requisição POST para a API
        response = requests.post(url, json=payload)
        
        # Verificar se a requisição foi bem-sucedida
        if response.status_code == 200:
            data = response.json()
            
            # Obter o total de páginas na primeira iteração
            if total_de_paginas is None:
                total_de_paginas = data.get('total_de_paginas', 1)
                print(f"Total de páginas: {total_de_paginas}")
            
            # Iterar sobre os registros retornados pela API
            for item in data.get('nfCadastro', []):
                for det in item.get('det', []):
                    # prod é um objeto, não um array
                    prod = det.get('prod')
                    if isinstance(prod, dict):
                        ncm = prod.get('NCM')
                        ucom = prod.get('uCom')
                        qcom = prod.get('qCom')
                        xprod = prod.get('xProd')
                        
                        # Escrever os dados extraídos no arquivo CSV
                        writer.writerow({'NCM': ncm, 'uCOM': ucom, 'qCom': qcom, 'xProd': xprod})
            
            print(f"Página {pagina} processada com sucesso - {len(data.get('nfCadastro', []))} registros")
            
            # Verificar se há próxima página
            if pagina >= total_de_paginas:
                break
            
            pagina += 1
        else:
            print(f"Erro na requisição: {response.status_code}")
            break

print("Dados de todas as páginas extraídos e armazenados em produtos.csv")

Total de páginas: 10
Página 1 processada com sucesso - 100 registros
Página 2 processada com sucesso - 100 registros
Página 3 processada com sucesso - 100 registros
Página 4 processada com sucesso - 100 registros
Página 5 processada com sucesso - 100 registros
Página 6 processada com sucesso - 100 registros
Página 7 processada com sucesso - 100 registros
Página 8 processada com sucesso - 100 registros
Página 9 processada com sucesso - 100 registros
Página 10 processada com sucesso - 23 registros
Dados de todas as páginas extraídos e armazenados em produtos.csv
